## 🎯 Objetivos de Aprendizaje

Al finalizar esta clase, el estudiante será capaz de:

1. **Modelar** con `scikit-fuzzy` el controlador difuso propuesto (ej. aire acondicionado).
2. **Definir** las variables de entrada/salida, sus funciones de membresía y el conjunto de reglas.
3. **Simular** el sistema y defuzzificar la salida para casos concretos.
4. **Interpretar** la superficie de control obtenida y evaluar el diseño.

> 🧪 **Cuaderno de desarrollo (grupo CSSJ):** resolución guiada del ejercicio del módulo.

> 💡 **Prerrequisitos:** [02_SciKit_Fuzzy](02_SciKit_Fuzzy.ipynb).

## Ejercicio 1: Aire Acondicionado. 

Ud. ha sido contratado por IPD-Electrónics para trabajar en su marca TEL-Air que comercializará equipos de aire acondicionado. Estos dispositivos climatizan ambientes interiores en el rango entre $17ºC$ a $30ºC$, lanzando aire frio o aire caliente dependiendo de las instrucciones que el usuario le entregue, hasta llegar a la temperatura deseada. Así:

* Para climatizar utilizando el modo _calor_, el aire acondicionado lanza aire caliente constante, a unos $42ºC$, hasta llegar a la temperatura deseada. 
* Para climatizar utilizando el modo _frío_, el aire acondiconado lanza aire fresco constante, a unos $13ºC$, hasta llegar a la temperatura deseada.
* Para mantener el clima actual del ambiente se utiliza el modo _stand by_, que provee de circulación del aire sin modificar la variación normal de la temperatura.
* Para climatizar automáticamente se utiliza el modo _auto_, el cual se encarga de escoger el modo del aire acondicionado, y el nivel de velocidad utilizado.

El aire es lanzado por los dispositivos en tres posibles velocidades: baja, media o alta. Estas velocidades modifican la temperatura, según el modo de aire y las caractiristicas de la zona climatizada, en los rangos (en $\frac{ºC}{\text{min}}$) $[0.1, 0.7]$, $[0.5, 1.2]$ y $[0.8, 1.9]$, respectivamente. Considere que si está en modo _stand by_ estas variaciones se darán para acercarse a la temperatura del exterior.  

Para controlar la temperatura del ambiente, los dispositivos cuentan con un termostato que les indica la temperatura actual de la zona climatizada.

Considere que para limitar el consumo energético, el modo _auto_ define las siguientes reglas bases:
* Si la diferencia de temperatura es negativa, debe utilizar modo calor.
* Si la diferencia de temperatura es positiva, debe utilizar modo frío.
* Si no hay diferencia de temperatura, debe utilizar el modo stand by.
* Si se encuentra cercano o en la temperatura deseada, no puede utilizar el modo de velocidad alta.
* Si se encuentra muy alejado en la tempratura, debe utilizar el modo de velocidad alta. 

Y que además, utilizando el historico de temperatura actual y de destino, reflejado en las configuraciones manuales, se ha determinado qué:
* La temperatura puede ser catalogada como fría si está bajo el umbral de los 15ºC.
* La temperatura puede ser catalogada como fresca si está en el rango de los 12.5ºC a los 20ºC.
* La temperatura puede ser catalogada como ambiente entre los 18ºC a los 25ºC.
* La temperatura puede ser catalogada como calurosa sobre el rango de los 23ºC.

Y como es un sistema basado en la percepción y el comfort humano, se definen las siguientes reglas básicas:
* Si la temperatura es fría, el aire debe utilizar modo calor. 
* Si la temperatura es calurosa, el aire debe utilizar modo frío.

1. Genere un sistema de inferencia difusa que permita para un instante determinado, utilizando el modo _auto_, determinar el modo de función del dispositivo y la velocidad con la cual lanza el aire. Determine claramente los antecendentes y los consecuentes del sistema, funciones de membresía y las reglas de inferencia difusa del sistema.

In [ ]:
# Incluir librerías
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

## Variables
### Antecedentes

In [ ]:
amb = ctrl.Antecedent(np.arange(0, 40.5, 0.5), 'ambiente') #frio, fresco, ambiente, caluroso
dif = ctrl.Antecedent(np.arange(-30, 23.5, 0.5), 'diferencia') #muy negativo, negativo, neutro, positivo, muy positivo

### Consecuentes

In [ ]:
vel = ctrl.Consequent(np.arange(0.1, 2.0, 0.1), "velocidad") # baja, media, alta
modo = ctrl.Consequent(np.arange(-1,1.01, 0.01), "modo") #frio, stand by, calor

# Membresías

In [ ]:
amb['frio'] = fuzz.zmf(amb.universe, 0,15)
amb['fresco'] = fuzz.pimf(amb.universe, 12.5, 15, 18, 20)
amb['ambiente'] = fuzz.gaussmf(amb.universe, 21, 1.5)
amb['caluroso'] = fuzz.smf(amb.universe, 23, 40)
amb.view()

In [ ]:
diffs = ['muy negativo', 'negativo', 'neutro', 'positivo', 'muy positivo']
dif.automf(names=diffs)
dif.view()

In [ ]:
vel['baja'] = fuzz.pimf(vel.universe, 0.0, 0.15, 0.65, 0.8)
vel['media'] = fuzz.pimf(vel.universe, 0.3, 0.45, 1.15, 1.3)
vel['alta'] = fuzz.pimf(vel.universe, 0.7, 0.85, 1.85, 2.0)
vel.view()

In [ ]:
modo['frio'] = fuzz.zmf(modo.universe,-1,0)
modo['stand by'] = fuzz.gaussmf(modo.universe, 0, 0.1) 
modo['calor'] = fuzz.smf(modo.universe, 0, 1)
modo.view()

Considere que para limitar el consumo energético, el modo _auto_ define las siguientes reglas bases:
* Si la diferencia de temperatura es negativa, debe utilizar modo calor.
* Si la diferencia de temperatura es positiva, debe utilizar modo frío.
* Si no hay diferencia de temperatura, debe utilizar el modo stand by.
* Si se encuentra cercano o en la temperatura deseada, no puede utilizar el modo de velocidad alta.
* Si se encuentra muy alejado en la tempratura, debe utilizar el modo de velocidad alta. 

Y como es un sistema basado en la percepción y el comfort humano, se definen las siguientes reglas básicas:
* Si la temperatura es fría, el aire debe utilizar modo calor. 
* Si la temperatura es calurosa, el aire debe utilizar modo frío.

In [ ]:
r1 = ctrl.Rule(dif['negativo'] | dif['muy negativo'], modo['calor'])
r2 = ctrl.Rule(dif['positivo'] | dif['muy positivo'], modo['frio'])
r3 = ctrl.Rule(dif['neutro'], modo['stand by'])
r4_1 = ctrl.Rule(dif['neutro'] | dif['negativo'] | dif['positivo'], vel['baja'])
r4_2 = ctrl.Rule(dif['neutro'] | dif['negativo'] | dif['positivo'], vel['media'])
r5 = ctrl.Rule(dif['muy negativo'] | dif['muy positivo'], vel['alta'])

r6 = ctrl.Rule(amb['frio'], modo['calor'])
r7 = ctrl.Rule(amb['caluroso'], modo['frio'])

ctrl_aire = ctrl.ControlSystem([r1, r2, r3, r4_1, r4_2, r5, r6, r7])

In [ ]:
AC = ctrl.ControlSystemSimulation(ctrl_aire)

t_ambiente = 20.0
t_destino = 17.0

AC.input['ambiente'] = t_ambiente
AC.input['diferencia'] = t_ambiente - t_destino

AC.compute()

In [ ]:
print(AC.output['modo'])
print(AC.output['velocidad'])
dif.view(sim=AC)
amb.view(sim=AC)
modo.view(sim=AC)
vel.view(sim=AC)

2. Simule un sistema de control difuso, que muestre como funcionaría el dispositivo durante 15 minutos, considerando una actualización de las condiciones cada 30 segundos.